Our prompt needs to assist users in writing three specific types of output for AWS use cases:

- Python code
- JSON configuration files
- Regular expressions

In [ ]:
from typing import Any

from anthropic import Anthropic
from anthropic.types import MessageParam
from dotenv import load_dotenv

load_dotenv()

Messages = list[MessageParam]

client: Anthropic = Anthropic()
model: str = "claude-sonnet-4-6"
json_system_prompt = "Respond only with raw valid JSON. No explanation, no markdown, no code blocks."
supported_formats_system_prompt = "Output only the raw solution. No explanation, no markdown, no code fences."

In [ ]:
def add_user_message(messages: Messages, text: str) -> None:
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: Messages, text: str) -> None:
    messages.append({"role": "assistant", "content": text})

def chat(messages: Messages, system_prompt: str|None = None, temperature: float = 1.0) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system_prompt is not None:
        params["system"] = system_prompt
    
    response = client.messages.create(**params)
    return response.content[0].text

In [ ]:
import json

def generate_dataset() -> list[dict[str, str]]:
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "<one of: python, json, regex>",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages: Messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system_prompt=json_system_prompt)
    return json.loads(text.strip())

In [ ]:
dataset = generate_dataset()

with open("dataset.local.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
def run_prompt(task: str) -> str:
    """Merges the prompt and test case input, then returns the result."""
    
    prompt = f""""
Please, solve the following task:

{task}

* Respond only with Python, JSON, or Regex
* Do not add any comments or commentary or explanation
    """
    messages: Messages = []
    add_user_message(messages, prompt)
    output = chat(messages, system_prompt=supported_formats_system_prompt)
    return output

In [ ]:
def grade_by_model(task: str, output: str, solution_criteria: str) -> dict[str, Any]:
    eval_prompt = f"""You are an expert code reviewer. Evaluate this AI-generated solution.

Original task:
<task>
{task}
</task>

Solution to evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{solution_criteria}
</criteria>

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10

Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages, system_prompt=json_system_prompt)
    return json.loads(eval_text)

In [ ]:
import ast
import re

def validate_json(text: str) -> int:
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text: str) -> int:
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text: str) -> int:
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response: str, response_format: str) -> int:
    if response_format == "json":
        return validate_json(response)
    elif response_format == "python":
        return validate_python(response)
    elif response_format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {response_format}")

In [ ]:
def run_test_case(test_case: dict[str, str]) -> dict[str, Any]:
    """Calls run_prompt, then grades the result."""
    
    task = test_case["task"]
    output = run_prompt(task)
    
    model_grade = grade_by_model(task, output, test_case["solution_criteria"])
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case["format"])

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean

def run_eval(dataset: list[dict[str, str]]) -> list[dict[str, Any]]:
    """Loads the dataset and calls run_test_case with each case."""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
with open("dataset.local.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))